In [81]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd


In [82]:
def create_webdriver():
    opts = Options()
    # opts.add_argument("--headless=new")  # enable for no-GUI scraping
    # opts.add_experimental_option("detach", True)  # keep window open after script ends
    return webdriver.Chrome(options=opts)


WEBSITE = "https://old.reddit.com/r/wallstreetbets/"
# https://old.reddit.com/r/wallstreetbets/
# https://books.toscrape.com/
driver = create_webdriver()
driver.get(WEBSITE)


In [83]:
# Wait until the project links are present
titles = []
post_idList = []
created_utcList = []
flairsList = []
# upvotesList = []
num_commentsList = []
permalinkList = []

i = 0

while i < 10:
    wait = WebDriverWait(driver, 10)
    things = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.thing"))
    )




    for thing in things:
        if "stickied" in thing.get_attribute("class"):
            continue
        if thing.get_attribute("data-promoted") == "true":
            continue
        title = thing.find_element(By.CSS_SELECTOR, 'p.title > a')
        titles.append(title.text)

        post_id = thing.get_attribute("data-fullname")
        post_idList.append(post_id)

        time_elements = thing.find_elements(By.TAG_NAME, "time")
        if time_elements:
            created_utc = time_elements[0].get_attribute("datetime")
        else:
            created_utc = None
        created_utcList.append(created_utc)

        flairs = thing.find_element(By.CSS_SELECTOR, 'span.linkflairlabel')
        flairsList.append(flairs.text)

        # upvotes = thing.find_element(By.CSS_SELECTOR, 'div.score')
        # upvotesList.append(upvotes.text)

        comments = thing.find_element(By.CSS_SELECTOR, 'a.comments')
        num_commentsList.append(comments.text)

        permalink = comments.get_attribute("href")
        permalinkList.append(permalink)

        


    
    next_button = WebDriverWait(driver, 10).until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'span.next-button')))
    if next_button.get_attribute('Disabled'):
        break  # Exit the loop if the next button is disabled
    else:
        # Click the next button to navigate to the next page
        next_button.click()
    i += 1


driver.quit()


## NOTE: Need to scrape other subreddits as well
- r/stocks
- r/personalfinance

In [85]:
df = pd.DataFrame({'titles': titles, 'post_idList': post_idList, 'created_utc': created_utcList, 'flairs': flairsList, 'num_comments': num_commentsList, 'permalink': permalinkList})
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 500)  # adjust for your screen
pd.set_option('display.max_columns', None)
print(df)

                                                                                                                                    titles post_idList                created_utc              flairs    num_comments                                                                                                     permalink
0                                                                                     The real price is the friends we made along the way.  t3_1n0nvtw  2025-08-26T14:48:44+00:00                Meme     46 comments     https://old.reddit.com/r/wallstreetbets/comments/1n0nvtw/the_real_price_is_the_friends_we_made_along_the/
1                                                                                                            I f*d up really bad this time  t3_1n0td6h  2025-08-26T18:12:22+00:00                Loss    108 comments                        https://old.reddit.com/r/wallstreetbets/comments/1n0td6h/i_fd_up_really_bad_this_time/
2                           

# Next Steps:
- data cleaning
- perform analytics
- visualizations

In [86]:
df['titles'] = df['titles'].str.replace('\n', ' ', regex=False).str.strip()
df['flairs'] = df['flairs'].fillna('').str.strip()
df = df[df['flairs'] != 'Daily Discussion']

In [87]:
df['num_comments'] = (df['num_comments'].str.extract('(\d+)').fillna(0).astype(int))
df['num_comments']

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/var/folders/49/dj3znqqj2zx9gx86y4p9jzgr0000gq/T/ipykernel_24056/1112965993.py:1: SyntaxWarning: invalid escape sequence '\d'
  df['num_comments'] = (df['num_comments'].str.extract('(\d+)').fillna(0).astype(int))


0        46
1       108
2      1188
3        44
4        69
5        70
6       103
8        18
9        13
10      175
11       11
12       30
13       16
14      130
15      222
16       76
17      410
18       10
19      159
20       13
21        5
22      199
24       37
25       90
26       69
27       14
28      312
29       35
30       82
31       36
32       30
33       83
34       85
35       26
36       51
38       26
39       79
40      171
41       29
42       89
43      268
44       37
45       12
46       79
47        9
48       20
49        9
50       40
51       97
52      958
54      197
55       19
56        8
57       61
58       20
59       43
60       24
61      486
62      147
63      329
64       64
65      379
66      113
67      342
68       68
69      721
70      214
71      151
72       47
73      110
74      763
75      530
76      107
77       42
78      185
79      140
80      261
81       47
82       82
83      256
84      448
85      168
86       94
87  

In [88]:
df['created_utc'] = pd.to_datetime(df['created_utc'], utc=True)
df['date'] = df['created_utc'].dt.date

In [ ]:
# list of all ticker symbols scraped
pattern = r'\b[A-Z]{1,5}\b'
df['tickers'] = df['titles'].str.findall(pattern)
df['tickers']


KeyError: 'title'

## need to grab nasdaq list and nyse list to compare ticker symbols with

In [90]:
# public source nasdaq lists, combined into one set
df = pd.read_csv("nasdaqlist.txt", sep="|")
df2 = pd.read_csv("nasdaqlistpt2.txt", sep="|")

# print(df.columns)
# print(df['NASDAQ Symbol'])
# print(df2['Symbol'])
tickers = set(df['NASDAQ Symbol'])
tickers2 = set(df2['Symbol'])
tickers_all = tickers.union(tickers2)
# tickers_all

In [91]:
# dictionary to store the number of occurrences of each ticker mentioned, after comparing it to the NASDAQ list to verify it is real
final = {}
for arr in df['tickers']:
	for value in arr:
		if value in tickers_all:
			if value not in final:
				final[value] = 1
			else:
				final[value] += 1
		else:
			continue
final

KeyError: 'tickers'

## next steps: create dashboard